# VentriQ — Phase 3: Dataset Preparation, Manifest & Split Reproducibility

**Objective:** Establish canonical record indexing, ensure complete traceability between ECG waveforms and diagnostic targets, verify zero patient leakage across partitions, and provide PyTorch-compatible dataset loaders without duplicating binary files.

### Key Tasks:
1. **Canonical Record Representation (`ECGRecord`):** Immutable schema with global ID, patient key, split, waveform slice ref, and diagnostic labels.
2. **Exclusion Rules & Audit Logging (`DatasetManifest`):** Cryptographic SHA-256 source checksums, audit breakdown, target distributions.
3. **Zero-Copy PyTorch Dataset (`VentriQTorchDataset`):** On-the-fly memory-mapped slicing, multi-worker safety, shape standardization `(12, 2500)`.
4. **Patient-Level Split Verification & Preservation (`src.data.splits`):** Detecting leakage, enforcing zero patient overlap across partitions and in $K$-Fold cross validation.

In [ ]:
from pathlib import Path
import sys

# Ensure project root is in sys.path
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader

from src.data.dataset import (
    ECGRecord,
    DatasetManifest,
    VentriQTorchDataset,
    ventriq_worker_init_fn,
)
from src.data.splits import (
    verify_patient_isolation,
    PatientGroupKFold,
    create_patient_kfold_splits,
)
from src.preprocessing.signal import ECGSignalPreprocessor
from src.utils.paths import get_raw_echonext_dir, REPORTS_DIR

print(f"Project Root: {PROJECT_ROOT}")
print(f"PyTorch Version: {torch.__version__}")

## 1. Load Canonical Dataset Manifest & Inspect Source Hashes

In [ ]:
manifest_path = REPORTS_DIR / "dataset_manifest.json"
if not manifest_path.exists():
    manifest = DatasetManifest.build_from_echonext(get_raw_echonext_dir())
    manifest.save(manifest_path)
else:
    manifest = DatasetManifest.load(manifest_path)

print(f"Total Retained Records: {len(manifest.records)}")
print("\nSource File Checksums (SHA-256):")
for fname, h in manifest.source_checksums.items():
    print(f"  {fname:30s} -> {h}")

print("\nRetained Records per Split:")
for sp, cnt in manifest.retained_counts_per_split.items():
    pats = manifest.patient_counts_per_split.get(sp, 0)
    print(f"  {sp:10s}: {cnt:4d} records ({pats:4d} unique patients)")

## 2. Verify Patient Isolation (Zero Leakage Across Splits)

In [ ]:
report = verify_patient_isolation(manifest.records, raise_on_leak=True)
print(report.summary())

## 3. PatientGroupKFold Cross-Validation Demonstration

In [ ]:
# Filter to train records for cross-validation demonstration
train_records = [r for r in manifest.records if r.split == "train"]
kfold = PatientGroupKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Demonstrating 5-Fold Patient-Level CV on {len(train_records)} training records:")
for fold, (train_idx, val_idx) in enumerate(kfold.split(train_records)):
    train_pats = set(train_records[i].patient_key for i in train_idx)
    val_pats = set(train_records[i].patient_key for i in val_idx)
    overlap = train_pats.intersection(val_pats)
    print(f"  Fold {fold+1}: Train={len(train_idx)} recs ({len(train_pats)} pats) | Val={len(val_idx)} recs ({len(val_pats)} pats) | Shared Patients={len(overlap)}")
    assert len(overlap) == 0, f"Leakage detected in fold {fold+1}!"

## 4. PyTorch Dataset & Multi-Worker DataLoader

In [ ]:
# Initialize PyTorch dataset directly from manifest with channels_first=(12, 2500)
torch_ds = VentriQTorchDataset(
    root=get_raw_echonext_dir(),
    records=manifest.records,
    split="train",
    channels_first=True,
)

sample = torch_ds[0]
print("Sample 0 retrieved:")
print(f"  Record ID:       {sample['record_id']}")
print(f"  Patient Key:     {sample['patient_key']}")
print(f"  Waveform Tensor: shape={sample['waveform'].shape}, dtype={sample['waveform'].dtype}")
print(f"  LVEF Continuous: {sample['lvef'].item():.1f}%")
print(f"  LVEF <= 45% Flag:{sample['lvef_lte_45'].item()}")
print(f"  SHD Flag:        {sample['shd'].item()}")

# Create multi-worker DataLoader
loader = DataLoader(
    torch_ds,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    worker_init_fn=ventriq_worker_init_fn,
)

first_batch = next(iter(loader))
print("\nFirst Batch Loaded via Multi-Worker DataLoader:")
print(f"  Batch Waveforms: shape={first_batch['waveform'].shape}")
print(f"  Batch LVEF:      {first_batch['lvef']}")
print(f"  Batch Records:   {first_batch['record_id']}")